From Fall 2022–2025, among Bay Area public high schools with UC Berkeley freshman applicants, how large is the difference between each school’s observed and expected admit rate, and which schools consistently perform above or below expectations?

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

Saving bay_area_modeling_table.csv to bay_area_modeling_table.csv
Saving dashboard_data.csv to dashboard_data.csv
Saving README.md to README.md
Saving uc_admissions_summary_by_ethnicity.csv to uc_admissions_summary_by_ethnicity.csv
Saving uc_freshman_admission_by_discipline.csv to uc_freshman_admission_by_discipline.csv
Saving uc_transfer_admission_by_major.csv to uc_transfer_admission_by_major.csv


In [ ]:
bay

In [ ]:


main = pd.read_csv("bay_area_modeling_table.csv", low_memory=False)
dashboard = pd.read_csv("dashboard_data.csv", low_memory=False)
ethnicity = pd.read_csv("uc_admissions_summary_by_ethnicity.csv")+
discipline = pd.read_csv("uc_freshman_admission_by_discipline.csv")
transfer = pd.read_csv("uc_transfer_admission_by_major.csv")

print("Main:", main.shape)
print("Dashboard:", dashboard.shape)
print("Ethnicity:", ethnicity.shape)
print("Discipline:", discipline.shape)
print("Transfer:", transfer.shape)

Main: (34311, 65)
Dashboard: (34311, 72)
Ethnicity: (4239, 6)
Discipline: (101, 12)
Transfer: (49, 13)


Q1

In [ ]:
y2025 = main[main["fall_term"] == 2025]

total_campus_applications = y2025[
    y2025["campus"] != "Universitywide"
]["applicants"].sum()

unique_applicants = y2025[
    y2025["campus"] == "Universitywide"
]["applicants"].sum()

answer = total_campus_applications / unique_applicants

print("Total campus applications:", total_campus_applications)
print("Unique applicants:", unique_applicants)
print("Answer:", round(answer, 2))

Total campus applications: 151755.0
Unique applicants: 26457.0
Answer: 5.74


Q2

In [ ]:
ucla = main[
    (main["fall_term"] == 2025) &
    (main["campus"] == "Los Angeles")
]

total_applicants = ucla["applicants"].sum()
total_admits = ucla["admits"].sum()

admit_rate = total_admits / total_applicants

print("Applicants:", total_applicants)
print("Admits:", total_admits)
print("Admit rate:", round(admit_rate * 100, 2), "%")

Applicants: 18516.0
Admits: 1514.0
Admit rate: 8.18 %


Q3

In [ ]:
cs = discipline[
    (discipline["fall_term"] == 2025) &
    (discipline["broad_discipline"] == "Computer Science")
][["campus", "admit_rate"]].rename(
    columns={"admit_rate": "cs_admit_rate"}
)

overall = discipline[
    (discipline["fall_term"] == 2025) &
    (discipline["broad_discipline"] == "All disciplines")
][["campus", "admit_rate"]].rename(
    columns={"admit_rate": "overall_admit_rate"}
)

comparison = cs.merge(overall, on="campus")

comparison["cost_pp"] = (
    comparison["overall_admit_rate"] -
    comparison["cs_admit_rate"]
) * 100

comparison = comparison.sort_values(
    "cost_pp",
    ascending=False
)

display(comparison)

winner = comparison.iloc[0]

print("Campus:", winner["campus"])
print("Overall admit rate:", winner["overall_admit_rate"] * 100, "%")
print("CS admit rate:", winner["cs_admit_rate"] * 100, "%")
print("Cost:", round(winner["cost_pp"], 2), "percentage points")

,campus,cs_admit_rate,overall_admit_rate,cost_pp
1,Davis,0.19,0.44,25.0
5,San Diego,0.20,0.28,8.0
4,Riverside,0.81,0.87,6.0
0,Berkeley,0.06,0.11,5.0
6,Santa Barbara,0.34,0.38,4.0
3,Los Angeles,0.07,0.09,2.0
2,Irvine,0.28,0.29,1.0
7,Santa Cruz,0.79,0.72,-7.0


Campus: Davis
Overall admit rate: 44.0 %
CS admit rate: 19.0 %
Cost: 25.0 percentage points


Q4

In [ ]:
berkeley_cs = discipline[
    (discipline["fall_term"] == 2025) &
    (discipline["campus"] == "Berkeley") &
    (discipline["broad_discipline"] == "Computer Science")
]

q1 = berkeley_cs["admit_gpa_p25"].iloc[0]
q3 = berkeley_cs["admit_gpa_p75"].iloc[0]

iqr = q3 - q1

print("25th percentile:", q1)
print("75th percentile:", q3)
print("IQR:", round(iqr, 2), "GPA points")

25th percentile: 4.2
75th percentile: 4.29
IQR: 0.09 GPA points


Q5

In [ ]:
eth2025 = ethnicity[
    (ethnicity["fall_term"] == 2025) &
    (ethnicity["entrant_level"] == "freshman")
]

counts = eth2025.pivot_table(
    index=["campus", "ethnicity"],
    columns="count_type",
    values="n",
    aggfunc="sum"
).reset_index()

counts["admit_rate"] = (
    counts["Adm"] / counts["App"]
)

comparison = counts[
    counts["ethnicity"].isin([
        "White",
        "Hispanic/Latino(a)"
    ])
].pivot(
    index="campus",
    columns="ethnicity",
    values="admit_rate"
)

# Remove Systemwide because the question asks about the 9 campuses
campus_only = comparison.drop(index="Systemwide")

campus_only["White_Higher"] = (
    campus_only["White"] >
    campus_only["Hispanic/Latino(a)"]
)

display(campus_only)

answer = campus_only["White_Higher"].sum()

print("Answer:", answer)

ethnicity,Hispanic/Latino(a),White,White_Higher
campus,,,
Berkeley,0.118083,0.120194,True
Davis,0.358684,0.450360,True
Irvine,0.186305,0.274835,True
Los Angeles,0.075304,0.099984,True
Merced,0.950819,0.969137,True
Riverside,0.832656,0.902254,True
San Diego,0.259083,0.279440,True
Santa Barbara,0.308506,0.381941,True
Santa Cruz,0.619164,0.791047,True


Answer: 9


Q6

In [ ]:
systemwide = counts[
    (counts["campus"] == "Systemwide") &
    (counts["ethnicity"].isin([
        "White",
        "Hispanic/Latino(a)"
    ]))
].copy()

systemwide["admit_rate_pct"] = (
    systemwide["admit_rate"] * 100
)

display(
    systemwide[
        ["ethnicity", "App", "Adm", "admit_rate_pct"]
    ]
)

winner = systemwide.loc[
    systemwide["admit_rate"].idxmax()
]

print("Higher group:", winner["ethnicity"])
print(
    "Admit rate:",
    round(winner["admit_rate_pct"], 2),
    "%"
)

count_type,ethnicity,App,Adm,admit_rate_pct
75,Hispanic/Latino(a),55624,41458,74.532576
79,White,38390,26368,68.684553


Higher group: Hispanic/Latino(a)
Admit rate: 74.53 %


Q7

In [ ]:
graduates_2023 = main[
    (main["fall_term"] == 2023) &
    (main["campus"] == "Universitywide")
]

ccc_students = graduates_2023["enrolled_ccc"].sum()
hs_completers = graduates_2023["hs_completers"].sum()

ccc_share = ccc_students / hs_completers

print("CCC students:", ccc_students)
print("HS completers:", hs_completers)
print("Share:", round(ccc_share * 100, 2), "%")

CCC students: 21644.0
HS completers: 64345.0
Share: 33.64 %


Q8

In [ ]:
mission_sj = main[
    (main["fall_term"] == 2023) &
    (main["campus"] == "Universitywide") &
    (main["high_school"] == "MISSION SAN JOSE HIGH SCHOOL")
]

applicants = mission_sj["applicants"].iloc[0]
ag_completers = mission_sj["ag_completers"].iloc[0]

share = applicants / ag_completers

print("UC applicants:", applicants)
print("a-g completers:", ag_completers)
print("Share:", round(share * 100, 2), "%")

UC applicants: 420.0
a-g completers: 424.0
Share: 99.06 %


Q9

In [ ]:
schools_2025 = main[
    (main["fall_term"] == 2025) &
    (main["campus"] == "Universitywide") &
    (main["applicants"] > 0)
]

answer = schools_2025["high_school"].nunique()

print("Distinct high schools:", answer)

Distinct high schools: 244


Q10

In [ ]:
choices = [
    "HERCULES HIGH SCHOOL",
    "MISSION SENIOR HIGH SCHOOL",
    "MONTEREY TRAIL HIGH SCHOOL",
    "PHILLIP & SALA BURTON ACAD HS"
]

berkeley_choices = dashboard[
    (dashboard["fall_term"].between(2022, 2025)) &
    (dashboard["campus"] == "Berkeley") &
    (dashboard["high_school"].isin(choices))
].copy()

results = (
    berkeley_choices
    .groupby("high_school")["admit_rate_residual"]
    .mean()
    .dropna()
    .sort_values(ascending=False)
)

display((results * 100).rename("Average Residual (percentage points)"))

winner = results.idxmax()
gap = results.max() * 100

print("Answer:", winner)
print("Average residual:", round(gap, 2), "percentage points")

,Average Residual (percentage points)
high_school,
MISSION SENIOR HIGH SCHOOL,24.996445
HERCULES HIGH SCHOOL,7.546225
PHILLIP & SALA BURTON ACAD HS,-10.638553


Answer: MISSION SENIOR HIGH SCHOOL
Average residual: 25.0 percentage points


In [ ]:
df = pd.read_csv("dashboard_data.csv", low_memory=False)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (34311, 72)


,fall_term,campus,atp_code,cds_code,high_school,city,county,cde_district,zip,lat,...,caaspp_mathematics_mean_score,caaspp_ela_pct_met,caaspp_mathematics_pct_met,expected_admit_rate,admit_rate_residual,peer_cohort_students,peer_ag_completers,peer_applicants,peer_admits,peer_enrollees
0,2005,Universitywide,52910,3.868478e+13,ABRAHAM LINCOLN HIGH SCHOOL,San Francisco,San Francisco,San Francisco Unified,94116-1723,37.747259,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005,Universitywide,53075,4.369666e+13,ABRAHAM LINCOLN HIGH SCHOOL,San Jose,Santa Clara,San Jose Unified,95126-2002,37.329212,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2005,Universitywide,51315,7.616301e+12,ACALANES HIGH SCHOOL,Lafayette,Contra Costa,Acalanes Union High,94549-2623,37.904313,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2005,Universitywide,53276,4.369674e+13,ADRIAN C WILCOX HIGH SCHOOL,Santa Clara,Santa Clara,Santa Clara Unified,95051-1153,37.366875,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2005,Universitywide,50011,1.611190e+12,ALAMEDA COMMUNITY LEARNING CTR,Alameda,Alameda,Alameda Unified,94501-1851,37.779051,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Beyond the GPA: UC Berkeley Admissions

## Research Question

##From Fall 2022–2025, among Bay Area public high schools applying to
##UC Berkeley, how large is the gap between observed and expected freshman
#admit rates, and which schools consistently have the largest positive or
##negative gaps?

### Population
#Bay Area public high schools with freshman applicants to UC Berkeley.

### Time Window
#Fall 2022 through Fall 2025.

### Primary Metric

#Admissions Residual = Observed Admit Rate − Expected Admit Rate

#A positive residual means a school's applicant pool was admitted at a
#higher rate than expected by the provided model. A negative residual
#means its admit rate was below the model expectation.

In [ ]:
important_columns = [
    "fall_term",
    "campus",
    "cds_code",
    "high_school",
    "city",
    "county",
    "applicants",
    "admits",
    "admit_rate",
    "applicant_gpa",
    "ag_completion_rate",
    "frpm_pct",
    "caaspp_mathematics_pct_met",
    "expected_admit_rate",
    "admit_rate_residual"
]

df[important_columns].head()

,fall_term,campus,cds_code,high_school,city,county,applicants,admits,admit_rate,applicant_gpa,ag_completion_rate,frpm_pct,caaspp_mathematics_pct_met,expected_admit_rate,admit_rate_residual
0,2005,Universitywide,3.868478e+13,ABRAHAM LINCOLN HIGH SCHOOL,San Francisco,San Francisco,172.0,145.0,0.843023,3.897500,NaN,NaN,NaN,NaN,NaN
1,2005,Universitywide,4.369666e+13,ABRAHAM LINCOLN HIGH SCHOOL,San Jose,Santa Clara,80.0,69.0,0.862500,3.946744,NaN,NaN,NaN,NaN,NaN
2,2005,Universitywide,7.616301e+12,ACALANES HIGH SCHOOL,Lafayette,Contra Costa,138.0,129.0,0.934783,3.980798,NaN,NaN,NaN,NaN,NaN
3,2005,Universitywide,4.369674e+13,ADRIAN C WILCOX HIGH SCHOOL,Santa Clara,Santa Clara,75.0,71.0,0.946667,3.851448,NaN,NaN,NaN,NaN,NaN
4,2005,Universitywide,1.611190e+12,ALAMEDA COMMUNITY LEARNING CTR,Alameda,Alameda,11.0,10.0,0.909091,3.862000,NaN,NaN,NaN,NaN,NaN


In [ ]:
berkeley = df[
    (df["campus"] == "Berkeley") &
    (df["fall_term"].between(2022, 2025))
].copy()

print("Rows:", len(berkeley))
print("Unique school names:", berkeley["high_school"].nunique())
print("Years:", sorted(berkeley["fall_term"].unique()))

Rows: 946
Unique school names: 241
Years: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [ ]:
berkeley[
    [
        "applicants",
        "admits",
        "admit_rate",
        "expected_admit_rate",
        "admit_rate_residual"
    ]
].isna().sum()

,0
applicants,0
admits,244
admit_rate,244
expected_admit_rate,288
admit_rate_residual,442


In [ ]:
berkeley["school_id"] = (
    berkeley["high_school"].fillna("") + " | " +
    berkeley["city"].fillna("")
)

berkeley[["school_id", "fall_term"]].head()

,school_id,fall_term
25295,ABRAHAM LINCOLN HIGH SCHOOL | San Francisco,2022
25296,ABRAHAM LINCOLN HIGH SCHOOL | San Jose,2022
25297,ACADEMY-SAN FRAN @ MCATEER | San Francisco,2022
25298,ACALANES HIGH SCHOOL | Lafayette,2022
25299,ACE CHARTER HIGH SCHOOL | San Jose,2022


In [ ]:
check = berkeley[
    ["admit_rate", "expected_admit_rate", "admit_rate_residual"]
].dropna().copy()

check["calculated_residual"] = (
    check["admit_rate"] - check["expected_admit_rate"]
)

check.head(10)

,admit_rate,expected_admit_rate,admit_rate_residual,calculated_residual
27524,0.142857,0.147320,-0.004463,-0.004463
27525,0.175676,0.154563,0.021113,0.021113
27527,0.200000,0.162777,0.037223,0.037223
27529,0.146154,0.150229,-0.004075,-0.004075
27532,0.116959,0.152068,-0.035109,-0.035109
27534,0.189781,0.167778,0.022003,0.022003
27535,0.176471,0.162234,0.014236,0.014236
27538,0.120805,0.143973,-0.023168,-0.023168
27539,0.125000,0.155411,-0.030411,-0.030411
27540,0.085799,0.142760,-0.056962,-0.056962


In [ ]:
(
    check["calculated_residual"] -
    check["admit_rate_residual"]
).abs().max()

1.249000902703301e-16

In [ ]:
print("Schools represented:", berkeley["school_id"].nunique())

print(
    "Total Berkeley applications:",
    int(berkeley["applicants"].sum())
)

print(
    "Years analyzed:",
    berkeley["fall_term"].min(),
    "to",
    berkeley["fall_term"].max()
)

Schools represented: 245
Total Berkeley applications: 78229
Years analyzed: 2022 to 2025


In [ ]:
overall_admit_rate = (
    berkeley["admits"].sum(skipna=True) /
    berkeley.loc[berkeley["admits"].notna(), "applicants"].sum()
)

print(f"Observed Berkeley admit rate: {overall_admit_rate:.2%}")

Observed Berkeley admit rate: 13.35%


In [ ]:
analysis = berkeley.dropna(
    subset=[
        "applicants",
        "admit_rate",
        "expected_admit_rate",
        "admit_rate_residual"
    ]
).copy()

print("Usable observations:", len(analysis))
print("Schools with usable residual data:", analysis["school_id"].nunique())

Usable observations: 504
Schools with usable residual data: 193


In [ ]:
analysis = berkeley.dropna(
    subset=[
        "applicants",
        "admit_rate",
        "expected_admit_rate",
        "admit_rate_residual"
    ]
).copy()

print("Usable observations:", len(analysis))
print("Schools with usable residual data:", analysis["school_id"].nunique())

Usable observations: 504
Schools with usable residual data: 193


In [ ]:
def summarize_school(g):
    total_apps = g["applicants"].sum()

    weighted_residual = np.average(
        g["admit_rate_residual"],
        weights=g["applicants"]
    )

    return pd.Series({
        "years_data": g["fall_term"].nunique(),
        "total_applicants": total_apps,
        "weighted_residual": weighted_residual,
        "weighted_residual_pp": weighted_residual * 100,
        "positive_years": (g["admit_rate_residual"] > 0).sum(),
        "negative_years": (g["admit_rate_residual"] < 0).sum(),
        "share_positive_years":
            (g["admit_rate_residual"] > 0).mean()
    })

school_summary = (
    analysis
    .groupby(
        ["school_id", "high_school", "city", "county"],
        dropna=False
    )
    .apply(summarize_school)
    .reset_index()
)

school_summary.head()

/tmp/ipykernel_5500/2458339907.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_school)


,school_id,high_school,city,county,years_data,total_applicants,weighted_residual,weighted_residual_pp,positive_years,negative_years,share_positive_years
0,ABRAHAM LINCOLN HIGH SCHOOL | San Francisco,ABRAHAM LINCOLN HIGH SCHOOL,San Francisco,San Francisco,3.0,573.0,0.002427,0.242734,1.0,2.0,0.333333
1,ABRAHAM LINCOLN HIGH SCHOOL | San Jose,ABRAHAM LINCOLN HIGH SCHOOL,San Jose,Santa Clara,3.0,200.0,0.019312,1.931163,2.0,1.0,0.666667
2,ACALANES HIGH SCHOOL | Lafayette,ACALANES HIGH SCHOOL,Lafayette,Contra Costa,3.0,344.0,0.011255,1.125501,2.0,1.0,0.666667
3,ADRIAN C WILCOX HIGH SCHOOL | Santa Clara,ADRIAN C WILCOX HIGH SCHOOL,Santa Clara,Santa Clara,2.0,257.0,-0.034785,-3.478469,0.0,2.0,0.000000
4,ALAMEDA COMMUNITY LEARNING CTR | Alameda,ALAMEDA COMMUNITY LEARNING CTR,Alameda,Alameda,1.0,13.0,0.002137,0.213652,1.0,0.0,1.000000


In [ ]:
MIN_APPLICANTS = 20

ranked = school_summary[
    school_summary["total_applicants"] >= MIN_APPLICANTS
].copy()

print("Schools after applicant threshold:", len(ranked))

Schools after applicant threshold: 180


In [ ]:
top_overperformers = (
    ranked
    .sort_values("weighted_residual_pp", ascending=False)
    [
        [
            "high_school",
            "city",
            "years_data",
            "total_applicants",
            "weighted_residual_pp",
            "positive_years"
        ]
    ]
    .head(15)
)

top_overperformers

,high_school,city,years_data,total_applicants,weighted_residual_pp,positive_years
117,MISSION SENIOR HIGH SCHOOL,San Francisco,3.0,244.0,25.052241,3.0
130,OAKLAND CHARTER HIGH SCHOOL,Oakland,3.0,139.0,10.848348,3.0
93,LEADERSHIP PUBLIC SCH RICHMOND,Richmond,3.0,96.0,9.895199,3.0
79,JAMES LICK HIGH SCHOOL,San Jose,2.0,31.0,8.906047,2.0
16,ANTIOCH HIGH SCHOOL,Antioch,2.0,67.0,8.060146,2.0
178,TERRA NOVA HIGH SCHOOL,Pacifica,1.0,23.0,7.913131,1.0
23,ASPIRE RICHMOND CA COLG PREP,Richmond,3.0,83.0,7.307869,3.0
48,DOZIER-LIBBEY MEDICAL HIGH SCH,Antioch,3.0,75.0,6.880564,2.0
71,HERCULES HIGH SCHOOL,Hercules,3.0,154.0,6.808455,2.0
53,EL CERRITO HIGH SCHOOL,El Cerrito,3.0,359.0,5.731855,3.0


In [ ]:
top_underperformers = (
    ranked
    .sort_values("weighted_residual_pp")
    [
        [
            "high_school",
            "city",
            "years_data",
            "total_applicants",
            "weighted_residual_pp",
            "negative_years"
        ]
    ]
    .head(15)
)

top_underperformers

,high_school,city,years_data,total_applicants,weighted_residual_pp,negative_years
89,KIPP SAN JOSE COLLEGIATE,San Jose,3.0,156.0,-10.879106,3.0
140,PHILLIP & SALA BURTON ACAD HS,San Francisco,3.0,218.0,-10.650342,3.0
170,SUMMIT PREPARATORY CHARTER HIG,Redwood City,2.0,62.0,-9.295221,2.0
155,SAN LEANDRO HIGH SCHOOL,San Leandro,3.0,324.0,-9.265165,3.0
15,ANN SOBRATO HIGH SCHOOL,Morgan Hill,3.0,230.0,-9.051797,3.0
72,HERITAGE HIGH SCHOOL,Brentwood,3.0,318.0,-9.015150,3.0
81,JEFFERSON HIGH SCHOOL,Daly City,3.0,142.0,-8.937051,3.0
183,WASHINGTON HIGH SCHOOL,Fremont,3.0,455.0,-8.928770,3.0
169,SOUTH SAN FRANCISCO HS,South San Francisco,3.0,121.0,-8.714607,3.0
57,FOOTHILL HIGH SCHOOL,Pleasanton,3.0,712.0,-8.451997,3.0


In [ ]:
consistent_over = ranked[
    (ranked["years_data"] >= 3) &
    (ranked["share_positive_years"] >= 0.75)
].sort_values(
    "weighted_residual_pp",
    ascending=False
)

consistent_over[
    [
        "high_school",
        "city",
        "years_data",
        "total_applicants",
        "weighted_residual_pp",
        "positive_years"
    ]
].head(15)

,high_school,city,years_data,total_applicants,weighted_residual_pp,positive_years
117,MISSION SENIOR HIGH SCHOOL,San Francisco,3.0,244.0,25.052241,3.0
130,OAKLAND CHARTER HIGH SCHOOL,Oakland,3.0,139.0,10.848348,3.0
93,LEADERSHIP PUBLIC SCH RICHMOND,Richmond,3.0,96.0,9.895199,3.0
23,ASPIRE RICHMOND CA COLG PREP,Richmond,3.0,83.0,7.307869,3.0
53,EL CERRITO HIGH SCHOOL,El Cerrito,3.0,359.0,5.731855,3.0
43,DE ANZA HIGH SCHOOL,Richmond,3.0,166.0,3.543599,3.0
128,NOVATO HIGH SCHOOL,Novato,3.0,173.0,2.602648,3.0
95,LELAND HIGH SCHOOL,San Jose,3.0,566.0,2.178177,3.0
145,PITTSBURG HIGH SCHOOL,Pittsburg,3.0,343.0,1.334323,3.0


In [ ]:
ranked["share_negative_years"] = (
    ranked["negative_years"] /
    ranked["years_data"]
)

consistent_under = ranked[
    (ranked["years_data"] >= 3) &
    (ranked["share_negative_years"] >= 0.75)
].sort_values(
    "weighted_residual_pp"
)

consistent_under[
    [
        "high_school",
        "city",
        "years_data",
        "total_applicants",
        "weighted_residual_pp",
        "negative_years"
    ]
].head(15)

,high_school,city,years_data,total_applicants,weighted_residual_pp,negative_years
89,KIPP SAN JOSE COLLEGIATE,San Jose,3.0,156.0,-10.879106,3.0
140,PHILLIP & SALA BURTON ACAD HS,San Francisco,3.0,218.0,-10.650342,3.0
155,SAN LEANDRO HIGH SCHOOL,San Leandro,3.0,324.0,-9.265165,3.0
15,ANN SOBRATO HIGH SCHOOL,Morgan Hill,3.0,230.0,-9.051797,3.0
72,HERITAGE HIGH SCHOOL,Brentwood,3.0,318.0,-9.015150,3.0
81,JEFFERSON HIGH SCHOOL,Daly City,3.0,142.0,-8.937051,3.0
183,WASHINGTON HIGH SCHOOL,Fremont,3.0,455.0,-8.928770,3.0
169,SOUTH SAN FRANCISCO HS,South San Francisco,3.0,121.0,-8.714607,3.0
57,FOOTHILL HIGH SCHOOL,Pleasanton,3.0,712.0,-8.451997,3.0
65,GRANADA HIGH SCHOOL,Livermore,3.0,317.0,-8.228178,3.0


In [ ]:
print("=== RESEARCH QUESTION RESULTS ===\n")

print(
    "Schools analyzed after threshold:",
    len(ranked)
)

print(
    "Consistent overperformers:",
    len(consistent_over)
)

print(
    "Consistent underperformers:",
    len(consistent_under)
)

if len(consistent_over) > 0:
    best = consistent_over.iloc[0]

    print("\nLargest consistent positive gap:")
    print(best["high_school"], "-", best["city"])
    print(
        f"{best['weighted_residual_pp']:+.2f} percentage points"
    )

if len(consistent_under) > 0:
    worst = consistent_under.iloc[0]

    print("\nLargest consistent negative gap:")
    print(worst["high_school"], "-", worst["city"])
    print(
        f"{worst['weighted_residual_pp']:+.2f} percentage points"
    )

=== RESEARCH QUESTION RESULTS ===

Schools analyzed after threshold: 180
Consistent overperformers: 9
Consistent underperformers: 57

Largest consistent positive gap:
MISSION SENIOR HIGH SCHOOL - San Francisco
+25.05 percentage points

Largest consistent negative gap:
KIPP SAN JOSE COLLEGIATE - San Jose
-10.88 percentage points


In [ ]:
top_names = consistent_over["school_id"].head(10)

yearly_top = (
    analysis[analysis["school_id"].isin(top_names)]
    .assign(residual_pp=lambda x: x["admit_rate_residual"] * 100)
    .pivot_table(
        index="school_id",
        columns="fall_term",
        values="residual_pp",
        aggfunc="mean"
    )
)

yearly_top

fall_term,2023,2024,2025
school_id,,,
ASPIRE RICHMOND CA COLG PREP | Richmond,10.219911,8.968970,2.559622
DE ANZA HIGH SCHOOL | Richmond,4.278675,4.907031,0.799073
EL CERRITO HIGH SCHOOL | El Cerrito,6.063296,4.103414,7.039752
LEADERSHIP PUBLIC SCH RICHMOND | Richmond,19.715971,4.758528,3.295703
LELAND HIGH SCHOOL | San Jose,5.191086,0.637434,0.721952
MISSION SENIOR HIGH SCHOOL | San Francisco,26.641612,20.287367,28.060356
NOVATO HIGH SCHOOL | Novato,0.918206,4.617559,2.615306
OAKLAND CHARTER HIGH SCHOOL | Oakland,3.005015,19.894792,15.124252
PITTSBURG HIGH SCHOOL | Pittsburg,2.723767,0.472014,0.650019


In [ ]:
characteristics = [
    "applicant_gpa",
    "ag_completion_rate",
    "frpm_pct",
    "caaspp_mathematics_pct_met",
    "caaspp_ela_pct_met",
    "grad_rate"
]

correlations = {}

for col in characteristics:
    temp = analysis[
        [col, "admit_rate_residual"]
    ].dropna()

    correlations[col] = temp[col].corr(
        temp["admit_rate_residual"]
    )

correlation_table = (
    pd.Series(correlations, name="correlation_with_residual")
    .sort_values(ascending=False)
)

correlation_table

,correlation_with_residual
frpm_pct,0.164985
ag_completion_rate,-0.016093
applicant_gpa,-0.081245
grad_rate,-0.153797
caaspp_mathematics_pct_met,-0.172764
caaspp_ela_pct_met,-0.176918


In [ ]:
# # Conclusion

# From Fall 2022–2025, UC Berkeley freshman admissions outcomes differed
# across Bay Area public high-school applicant pools even after considering
# the expected admit rates provided in the modeling dataset.

# Using an applicant-weighted admissions residual, we identified schools
# that repeatedly performed above or below their modeled expectations.

# We classified a school as a "consistent overperformer" when it had at
# least three years of usable observations and a positive admissions
# residual in at least 75% of those years.

# The residual represents:

# Observed Admit Rate − Expected Admit Rate

# and is reported in percentage points.

# These results describe school-level applicant pools and should not be
# interpreted as evidence that attending a particular high school causes
# an individual applicant to be admitted or rejected.